In [0]:
!pip install -q gpxpy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys, subprocess, importlib.util, os, textwrap, json, math
import math
import requests
import gpxpy
import pandas as pd
from datetime import datetime, timedelta, timezone

In [0]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2-lat1)
    dl = math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def load_gpx(path):
    with open(path, "r", encoding="utf-8") as f:
        gpx = gpxpy.parse(f)

    rows = []
    for track in gpx.tracks:
        for segment in track.segments:
            for p in segment.points:
                rows.append({
                    "lat": p.latitude,
                    "lon": p.longitude,
                    "time": p.time
                })

    if not rows:
        raise ValueError("Aucun point GPS trouvé dans le GPX.")

    df = pd.DataFrame(rows)

    # Distance cumulée
    distances = [0.0]
    for i in range(1, len(df)):
        distances.append(
            distances[-1] + haversine_km(
                df.iloc[i-1].lat, df.iloc[i-1].lon,
                df.iloc[i].lat, df.iloc[i].lon
            )
        )
    df["distance_km"] = distances
    return df

In [0]:
GPX_FILE = "data.gpx"
route = load_gpx(GPX_FILE)
route = route[route["distance_km"] < 100]
route.head()

,lat,lon,time,distance_km
0,45.660792,6.370067,2026-06-12 04:30:46.052000+00:00,0.000000
1,45.660661,6.369844,2026-06-12 04:30:49.589000+00:00,0.022639
2,45.660542,6.369601,2026-06-12 04:30:53.228000+00:00,0.045698
3,45.660358,6.369257,2026-06-12 04:30:58.661000+00:00,0.079363
4,45.660328,6.369199,2026-06-12 04:30:59.550000+00:00,0.084971


In [0]:
def sample_route(df, every_km=5):
    """Réduit le nombre de requêtes API : ~1 point tous les N km."""
    targets = list(range(0, int(df.distance_km.max()) + 1, every_km))
    indices = []
    for d in targets:
        idx = (df.distance_km - d).abs().idxmin()
        indices.append(idx)
    return df.loc[sorted(set(indices))].copy()

In [0]:
sampled_route = sample_route(route, every_km=5)
sampled_route.head()

,lat,lon,time,distance_km
0,45.660792,6.370067,2026-06-12 04:30:46.052000+00:00,0.000000
86,45.631876,6.331131,2026-06-12 04:41:10.708000+00:00,5.026073
155,45.603651,6.282634,2026-06-12 04:51:50.245000+00:00,10.092448
194,45.567160,6.254603,2026-06-12 05:02:52.315000+00:00,15.019744
329,45.558042,6.291019,2026-06-12 05:20:14.313000+00:00,19.992202


In [0]:
def query_overpass_water(points, radius_m=1500):
    coords = [(r.lat, r.lon) for _, r in points.iterrows()]

    clauses = []
    for lat, lon in coords:
        clauses.append(
            f'node["amenity"="drinking_water"]'
            f'(around:{radius_m},{lat},{lon});'
        )

    query = f"""
    [out:json][timeout:60];
    (
        {"".join(clauses)}
    );
    out body;
    """

    url = "https://overpass-api.de/api/interpreter"

    headers = {
        "User-Agent": "CyclingWaterPrototype/0.1"
    }

    response = requests.post(
        url,
        data={"data": query},
        headers=headers,
        timeout=120
    )

    response.raise_for_status()

    elements = response.json().get("elements", [])

    water = []

    for e in elements:
        tags = e.get("tags", {})

        if (tags.get("access") is not None):

            water.append({
                "osm_id": e.get("id"),
                "lat": e.get("lat"),
                "lon": e.get("lon"),
                "name": tags.get("name"),
                "drinking_water": tags.get("drinking_water"),
                "fee": tags.get("fee"),
                "access": tags.get("access"),
            })

    if not water:
        return pd.DataFrame()

    return pd.DataFrame(water).drop_duplicates("osm_id")

In [0]:
water_df = query_overpass_water(sampled_route)

In [0]:
def attach_route_distance(water_df, route_df):
    if water_df.empty:
        return water_df

    distances = []
    for _, w in water_df.iterrows():
        d = (
            (route_df["lat"] - w["lat"])**2 +
            (route_df["lon"] - w["lon"])**2
        )
        idx = d.idxmin()
        distances.append(route_df.loc[idx, "distance_km"])

    water_df = water_df.copy()
    water_df["route_distance_km"] = distances
    return water_df.sort_values("route_distance_km")

In [0]:
water_df = attach_route_distance(water_df, sampled_route)

In [0]:
def open_meteo(lat, lon, start_date=None):
    """Prévisions horaires météo au point donné."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join([
            "temperature_2m",
            "precipitation",
            "precipitation_probability",
            "wind_speed_10m",
            "wind_direction_10m",
            "weather_code"
        ]),
        "timezone": "auto",
        "forecast_days": 2
    }

    r = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params=params,
        timeout=20
    )
    r.raise_for_status()
    data = r.json()

    return pd.DataFrame({
        "time": pd.to_datetime(data["hourly"]["time"]),
        "temperature_C": data["hourly"]["temperature_2m"],
        "precipitation_mm": data["hourly"]["precipitation"],
        "rain_probability_%": data["hourly"]["precipitation_probability"],
        "wind_kmh": data["hourly"]["wind_speed_10m"],
        "wind_direction_deg": data["hourly"]["wind_direction_10m"],
        "weather_code": data["hourly"]["weather_code"],
    })

In [0]:
water_df

,osm_id,lat,lon,name,drinking_water,fee,access,route_distance_km
18,12848529644,45.662028,6.369813,None,None,no,yes,0.000000
15,12154726710,45.660582,6.368907,None,None,no,yes,0.000000
13,11379645343,45.666926,6.378570,None,None,no,yes,0.000000
16,12273127068,45.557209,6.250889,None,None,no,yes,15.019744
1,6936035706,45.530366,6.319018,None,None,None,yes,25.017426
17,12407589440,45.417528,6.268652,None,None,no,yes,39.965390
0,6877713894,45.344868,6.290916,None,None,no,yes,49.960955
5,7886529210,45.333144,6.308783,None,None,no,yes,49.960955
4,7886491091,45.341638,6.293456,None,None,no,yes,49.960955
6,7886529211,45.332005,6.310947,None,None,None,yes,49.960955


In [0]:
weather_results = []

for _, p in water_df.iterrows():
    try:
        w = open_meteo(p.lat, p.lon)
        # On prend la météo actuelle / prochaine heure comme aperçu.
        row = w.iloc[0].to_dict()
        row['osm_id'] = p['osm_id']
        weather_results.append(row)
    except Exception as e:
        print("Erreur météo:", e)

weather_df = pd.DataFrame(weather_results)

In [0]:
final_df = pd.merge(water_df, weather_df, on="osm_id")

In [0]:
final_df

,osm_id,lat,lon,name,drinking_water,fee,access,route_distance_km,time,temperature_C,precipitation_mm,rain_probability_%,wind_kmh,wind_direction_deg,weather_code
0,12848529644,45.662028,6.369813,None,None,no,yes,0.000000,2026-08-21,20.5,3.0,20,4.9,197,95
1,12154726710,45.660582,6.368907,None,None,no,yes,0.000000,2026-08-21,20.5,3.0,20,4.9,197,95
2,11379645343,45.666926,6.378570,None,None,no,yes,0.000000,2026-08-21,19.7,0.9,20,8.4,200,95
3,12273127068,45.557209,6.250889,None,None,no,yes,15.019744,2026-08-21,19.6,0.6,33,7.9,164,80
4,6936035706,45.530366,6.319018,None,None,None,yes,25.017426,2026-08-21,20.8,0.3,50,3.1,324,80
5,12407589440,45.417528,6.268652,None,None,no,yes,39.965390,2026-08-21,19.7,0.2,40,2.2,90,80
6,6877713894,45.344868,6.290916,None,None,no,yes,49.960955,2026-08-21,20.4,0.2,40,1.4,360,80
7,7886529210,45.333144,6.308783,None,None,no,yes,49.960955,2026-08-21,20.4,0.2,43,1.4,360,80
8,7886491091,45.341638,6.293456,None,None,no,yes,49.960955,2026-08-21,20.3,0.2,40,1.4,360,80
9,7886529211,45.332005,6.310947,None,None,None,yes,49.960955,2026-08-21,20.4,0.2,43,1.4,360,80
